In [1]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

model = ESMC.from_pretrained("esmc_300m").to("cuda") # or "cpu"

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
from concurrent.futures import ThreadPoolExecutor
from typing import Sequence

from esm.sdk.api import (
    ESM3InferenceClient,
    ESMProtein,
    ESMProteinError,
    LogitsConfig,
    LogitsOutput,
    ProteinType,
)

EMBEDDING_CONFIG = LogitsConfig(
    sequence=True, return_embeddings=True, return_hidden_states=True
)

def embed_sequence(model: ESM3InferenceClient, sequence: str) -> LogitsOutput:
    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein)
    output = model.logits(protein_tensor, EMBEDDING_CONFIG)
    return output


In [3]:
import pandas as pd
df = (
    pd.read_csv("scratch/grand_list/grand_list.tsv", sep = "\t")
#    .query("GeneFamily in ['LargeMAF', 'Outgroup']")
#    .query("GeneGroup != 'Ignore'")
)
print(df)

     IsOrig                                    ProteinID  ProteinLen  \
0       Yes              acornWorm__Dmbx__NP_001161526.1       375.0   
1       Yes              acornWorm__drgx__NP_001158481.1       328.0   
2       Yes      acornWorm__LOC100369471__XP_002735289.1       310.0   
3       Yes      acornWorm__LOC100374666__XP_002732304.1       170.0   
4       Yes      acornWorm__LOC102804715__XP_006821810.1       127.0   
...     ...                                          ...         ...   
2520    Yes  zokor__Unknown__TRINITY_DN51399_c0_g1_i1.p1       122.0   
2521    Yes  zokor__Unknown__TRINITY_DN75680_c0_g1_i1.p2       135.0   
2522    Yes    zokor__Unknown__TRINITY_DN848_c1_g1_i2.p1       154.0   
2523    Yes  zokor__Unknown__TRINITY_DN94490_c0_g1_i1.p1        99.0   
2524    Yes  zokor__Unknown__TRINITY_DN94597_c0_g1_i1.p1       104.0   

        TPM_max OrganismShortName OrganismColor ProteinSource  \
0           NaN         acornWorm  Nonchordates     Annotated   
1    

In [4]:
from tqdm import tqdm
import torch

# Assume df is your DataFrame with columns: 'ProteinID' and 'ProteinSeq'
# Make sure ProteinID is unique! If not, consider using index or another unique key.

embeddings_dict = {}

for protein_id, seq in tqdm(zip(df.ProteinID, df.ProteinSeq), total=len(df)):
    output = embed_sequence(model, seq)  # your existing function
    
    # Compute mean across sequence length (dim=-2), squeeze unnecessary dims
    mean_embedding = torch.mean(output.hidden_states, dim=-2).squeeze()  # [num_layers, hidden_size]
    
    # Convert to float32 and move to CPU early (safe for saving and later analysis)
    mean_embedding = mean_embedding.to(torch.float32).cpu()
    
    # Store in dictionary
    embeddings_dict[protein_id] = mean_embedding

# Save the entire dictionary to a file
save_path = 'mean_embeddings_by_protein.pt'
torch.save(embeddings_dict, save_path)

print(f"Saved mean embeddings for {len(embeddings_dict)} proteins to {save_path}")

100%|██████████| 2525/2525 [03:37<00:00, 11.62it/s]


Saved mean embeddings for 2525 proteins to mean_embeddings_by_protein.pt
